# Neural-network robustness certification (auto_LiRPA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/fmaiv/blob/main/day04/examples/nn/robustness.ipynb)

**FMAIV Day 4 — the frontier hands-on.** Same *verification* question as the rest of the week
("can the bad thing happen?"), now for a neural network:

> Within an L-infinity ball of radius `eps` around an input `x0`, can the classifier's prediction change?

We use **auto_LiRPA** — the CROWN bound-propagation engine underneath
[alpha,beta-CROWN](https://github.com/Verified-Intelligence/alpha-beta-CROWN), the VNN-COMP winner.
It computes a **certified** lower bound on the margin `z[true] - z[other]` over the *entire* ball.
If that bound is `> 0`, **no** input in the ball is misclassified — a proof, not a sample.

Everything here is tiny and **CPU-only** (a 2-D, 2-class MLP), so it runs in ~1s on the free Colab
tier. The same code applies unchanged to a trained MNIST/CIFAR network — see our
[AAAI'26 VNN-COMP tutorial](https://vnn-comp.github.io/#aaai2026) for full-scale notebooks.


## 0. Install

On Colab `torch` is already present, so we only add `auto_LiRPA`.
(Locally: `pip install torch --index-url https://download.pytorch.org/whl/cpu` then `pip install auto_LiRPA`.)


In [ ]:
!pip -q install auto_LiRPA

## 1. A tiny dataset and model
Two well-separated Gaussian blobs (2 classes) and a small ReLU MLP. Deterministic via a fixed seed.


In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)

def make_data(n=600):
    half = n // 2
    blob0 = torch.randn(half, 2) * 0.7 + torch.tensor([1.0, 1.0])   # class 0
    blob1 = torch.randn(half, 2) * 0.7 + torch.tensor([-1.0, -1.0]) # class 1 (some overlap)
    X = torch.cat([blob0, blob1], 0)
    y = torch.cat([torch.zeros(half), torch.ones(half)]).long()
    return X, y

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2,16), nn.ReLU(),
                                 nn.Linear(16,16), nn.ReLU(),
                                 nn.Linear(16,2))
    def forward(self, x):
        return self.net(x)

X, y = make_data()
model = MLP()
opt = torch.optim.Adam(model.parameters(), lr=0.05)
lossf = nn.CrossEntropyLoss()
for _ in range(300):
    opt.zero_grad(); lossf(model(X), y).backward(); opt.step()
model.eval()
acc = (model(X).argmax(1) == y).float().mean().item()
print(f'train accuracy: {acc:.3f}')

## 2. Certify the margin under an L-inf perturbation
`compute_bounds(..., C=C, method='CROWN')` returns a certified lower bound on the linear
specification `C @ logits`. We set `C` to pick out the margin `z[true] - z[other]`.
A lower bound `> 0` means **certified robust** at that `eps`.


In [ ]:
from auto_LiRPA import BoundedModule, BoundedTensor
from auto_LiRPA.perturbations import PerturbationLpNorm

# near-boundary correctly-classified point (small margin), where robustness breaks
with torch.no_grad():
    logits = model(X)
    correct = logits.argmax(1) == y
    margins = torch.where(correct,
        logits.gather(1, y.view(-1,1)).squeeze(1) - logits.gather(1, (1-y).view(-1,1)).squeeze(1),
        torch.full_like(logits[:,0], float('inf')))
    idx = int((margins - 1.5).abs().argmin())
x0 = X[idx:idx+1]
true_cls = y[idx].item()
other = 1 - true_cls
lirpa_model = BoundedModule(model, torch.empty_like(x0))

C = torch.zeros(1, 1, 2)
C[0, 0, true_cls] = 1.0
C[0, 0, other] = -1.0

print(f'input = {[round(v,3) for v in x0.tolist()[0]]}, true class = {true_cls}')
print(f"{'eps':>6} {'cert. margin':>13}  verdict")
for eps in [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]:
    ptb = PerturbationLpNorm(norm=float('inf'), eps=eps)
    bx = BoundedTensor(x0, ptb)
    lb, ub = lirpa_model.compute_bounds(x=(bx,), C=C, method='CROWN')
    m = lb.item()
    verdict = 'CERTIFIED ROBUST' if m > 0 else 'not certified by CROWN'
    print(f'{eps:6.2f} {m:13.4f}  {verdict}')

## 3. Visualize the decision boundary and an eps-ball
The shaded regions are the network's prediction; the star is `x0`; the box is the L-inf
`eps`-ball. When the whole box stays in one color, the point is robust at that `eps`.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
xs = np.linspace(-3.5, 3.5, 200)
gx, gy = np.meshgrid(xs, xs)
grid = torch.tensor(np.stack([gx.ravel(), gy.ravel()], 1), dtype=torch.float32)
with torch.no_grad():
    pred = model(grid).argmax(1).numpy().reshape(gx.shape)
plt.figure(figsize=(5,5))
plt.contourf(gx, gy, pred, alpha=0.25, levels=1)
plt.scatter(X[:,0], X[:,1], c=y, s=8, cmap='coolwarm')
px, py = x0[0].tolist()
plt.scatter([px],[py], marker='*', s=300, edgecolor='k', c='yellow', zorder=5)
eps = 0.1
plt.gca().add_patch(plt.Rectangle((px-eps, py-eps), 2*eps, 2*eps,
                                  fill=False, edgecolor='k', lw=2))
plt.title(f'decision regions; star = x0; box = L-inf ball, eps={eps}')
plt.xlabel('x1'); plt.ylabel('x2'); plt.show()

## Takeaways
- **Certified `> 0`** = a *proof* of robustness over the whole ball (sound; like CBMC/Lean verdicts).
- **"not certified"** does NOT mean "vulnerable": CROWN is *incomplete*, so the bound can dip below 0
  while the point is still safe. Complete verifiers (alpha,beta-CROWN) close that gap with branch-and-bound.
- This is the same soundness/completeness story as the rest of the course, now for NNs.

**Next:** the script form is `robustness.py` (and `robustness_starter.py` blanks the `compute_bounds`
call). For full-scale MNIST/CIFAR verification and competition benchmarks, see the
[AAAI'26 VNN-COMP tutorial](https://vnn-comp.github.io/#aaai2026).
